# ROC-AUC Says 0.95 and the Model Is Unusable

**Topic:** ROC vs precision-recall &nbsp;·&nbsp; **Difficulty:** Medium

ROC-AUC is invariant to class balance, which sounds like a feature and is
frequently a trap. At a 0.5% positive rate, a model can have a beautiful ROC
curve and still return more than five false alarms for every real hit, because the false
positive *rate* is tiny while the false positive *count* is enormous.

Precision-recall does not hide this. Compute both from scratch on the same
scores.

## The data (created in the next cell)

- `y_true` `(20000,)`: 110 actual positives (0.55%, drawn at a 0.5% rate).
- `scores` `(20000,)`: a decent but imperfect model's predicted probabilities.

## Your task

With **NumPy only** (no `sklearn.metrics`):

1. Sort by score descending and sweep the threshold through every distinct score.
2. `roc_auc`: area under the ROC curve (FPR on x, TPR on y) by the trapezoid
   rule.
3. `pr_auc`: average precision, computed as
   `sum over thresholds of (recall_i - recall_{i-1}) * precision_i`.
4. `precision_at_recall_50`: the highest precision achievable at recall >= 0.5.
5. `fp_per_tp_at_recall_50`: false positives per true positive at that operating
   point.
6. Set `answers` to a dict with those four values plus `"base_rate"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6, and cross-checked
against `sklearn.metrics` inside the evaluation cell.

In [ ]:
import numpy as np

rng = np.random.default_rng(3012)
n = 20000
y_true = (rng.random(n) < 0.005).astype(int)
scores = rng.beta(2, 8, n) + y_true * rng.beta(5, 3, n) * 0.55
scores = np.clip(scores, 0, 1)
print(f"n = {n}, positives = {y_true.sum()} ({y_true.mean():.4%})")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("roc_auc", [], 0.9472306778189131, -0.15860242426539126, 1e-06),
    ("pr_auc", [], 0.28047808231582655, -0.04696269330193968, 1e-06),
    ("precision_at_recall_50", [], 0.1510989010989011, -0.025299700040652344, 1e-06),
    ("fp_per_tp_at_recall_50", [], 5.618181818181818, -0.940697210503277, 1e-06),
    ("base_rate", [], 0.0055, -0.0009209090814797453, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "roc_auc" in answers:
    from sklearn.metrics import roc_auc_score, average_precision_score
    _r = float(roc_auc_score(y_true, scores))
    _a = float(average_precision_score(y_true, scores))
    _m = abs(answers["roc_auc"] - _r) < 1e-6 and abs(answers["pr_auc"] - _a) < 1e-6
    _gap = answers["roc_auc"] > 0.8 and answers["pr_auc"] < 0.45
    print(f"  {'matches sklearn':<24} {'ok' if _m else 'wrong'}  "
          f"(roc {answers['roc_auc']:.4f} vs {_r:.4f}, pr {answers['pr_auc']:.4f} vs {_a:.4f})")
    print(f"  {'ROC flatters, PR does not':<24} {'ok' if _gap else 'wrong'}")
    _ok = _ok and _m and _gap

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

ROC-AUC compares TPR against FPR, and FPR has the *number of negatives* in its
denominator (19,890 of them here), so a thousand false positives barely moves it.
Precision has the number of *predicted* positives in its denominator, which is
dominated by those same false positives, so it collapses. When the positive class
is rare and false alarms are expensive, precision-recall is the honest curve and
ROC-AUC is a number you can quote without learning anything.

```python
import numpy as np

order = np.argsort(-scores)
y = y_true[order]

tp = np.cumsum(y)
fp = np.cumsum(1 - y)
P = int(y_true.sum())
N = int(len(y_true) - P)

tpr = tp / P
fpr = fp / N
recall = tpr
precision = tp / np.maximum(tp + fp, 1)

roc_auc = float(np.trapezoid(np.r_[0, tpr], np.r_[0, fpr])) if hasattr(np, "trapezoid") \
    else float(np.trapz(np.r_[0, tpr], np.r_[0, fpr]))

rec_prev = np.r_[0.0, recall[:-1]]
pr_auc = float(np.sum((recall - rec_prev) * precision))

mask = recall >= 0.5
idx = int(np.flatnonzero(mask)[np.argmax(precision[mask])])
p50 = float(precision[idx])

answers = {
    "roc_auc": roc_auc,
    "pr_auc": pr_auc,
    "precision_at_recall_50": p50,
    "fp_per_tp_at_recall_50": float(fp[idx] / max(tp[idx], 1)),
    "base_rate": float(y_true.mean()),
}
```